# Complex Log Image Pipeline

This notebook makes the image-to-complex workflow explicit and inspectable. The goal is to separate the mathematical steps so you can study the geometry, change one parameter at a time, and keep the transformation pipeline modular.

We will use the principal-branch pipeline

$$z = \frac{x-x_0}{s} + i\frac{y_0-y}{s}, \qquad w = \log(z), \qquad w' = a w + b, \qquad z' = \exp(w').$$

Interpretation:

- `a` changes how the log-plane itself is stretched and rotated.
- `b` translates the log-plane, which becomes multiplication by `exp(b)` after we return to the `z`-plane.
- Because we use the principal branch of `log`, the output preview has a branch cut.

In [ ]:
import importlib
import importlib.util
import sys
from pathlib import Path

notebook_dir = Path.cwd()
requirements_path = notebook_dir / "requirements.txt"
required_modules = {
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "PIL": "pillow",
}

print("Kernel executable:", sys.executable)
missing = [pip_name for module_name, pip_name in required_modules.items() if importlib.util.find_spec(module_name) is None]

if missing:
    print("Missing packages in THIS notebook kernel:", ", ".join(missing))
    if requirements_path.exists():
        print("Run this in a new notebook cell, then restart the kernel:")
        print(f'%pip install -r "{requirements_path}"')
    raise RuntimeError("This notebook kernel is not using a Python environment with the required packages installed.")

import matplotlib.pyplot as plt
import numpy as np

if str(notebook_dir) not in sys.path:
    sys.path.append(str(notebook_dir))

import complex_log_image_pipeline as cip
cip = importlib.reload(cip)

plt.rcParams["figure.dpi"] = 130
plt.rcParams["axes.facecolor"] = "#0f172a"
plt.rcParams["savefig.facecolor"] = "white"

## 1. Choose An Image And The Source-Plane Geometry

The first choice is how pixel positions become complex numbers. We pick:

- an origin `(x0, y0)` in pixel coordinates,
- a scale `s` in pixels per complex unit,
- a small inner cutoff to avoid taking `log(0)`.

In [ ]:
IMAGE_PATH = None
# IMAGE_PATH = Path("my_image.png")

if IMAGE_PATH is None:
    image = cip.make_demo_image()
    image_label = "generated demo"
else:
    image = cip.load_image(IMAGE_PATH, max_size=900)
    image_label = Path(IMAGE_PATH).name

height, width = image.shape[:2]
origin_px = (width / 2.0, height / 2.0)
pixels_per_unit = min(height, width) / 2.25
inner_cutoff_px = 3.0

print(cip.describe_mapping(image.shape, origin_px, pixels_per_unit, inner_cutoff_px=inner_cutoff_px))

In [ ]:
parts = cip.principal_log_decomposition(image.shape, origin_px, pixels_per_unit, inner_cutoff_px=inner_cutoff_px)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))

cip.plot_image(axes[0], image, f"Source image ({image_label})")
cip.annotate_source_plane(axes[0], image.shape, origin_px, pixels_per_unit)

axes[1].imshow(np.log(np.maximum(parts["radius"], parts["min_radius"])), cmap="magma")
axes[1].set_title("log |z| over the source grid")
axes[1].set_xticks([])
axes[1].set_yticks([])

axes[2].imshow(parts["angle"], cmap="twilight", vmin=-np.pi, vmax=np.pi)
axes[2].set_title("arg(z) over the source grid")
axes[2].set_xticks([])
axes[2].set_yticks([])

plt.tight_layout()

## 2. Render The Principal Log Plane

To visualize `w = log(z)`, we sample a rectangular window in the `w`-plane. Each point `w = u + iv` is sent back to the source image through `z = exp(w)`.

This is the cleanest place to see why the logarithm turns radial structure into horizontal structure and angular structure into vertical structure.

In [ ]:
log_preview = cip.render_log_plane_preview(
    image,
    origin_px,
    pixels_per_unit,
    output_shape=(360, 480),
    inner_cutoff_px=inner_cutoff_px,
)

u0, u1 = log_preview["u_range"]
v0, v1 = log_preview["v_range"]

fig, ax = plt.subplots(figsize=(8.4, 5.2))
ax.imshow(log_preview["image"], extent=(u0, u1, v0, v1), origin="lower")
ax.set_title("Principal log-plane preview")
ax.set_xlabel("Re(w) = ln |z|")
ax.set_ylabel("Im(w) = arg(z)")
ax.grid(color="white", alpha=0.15)

## 3. Apply A Complex Operation In Log Space

We now choose

$$w' = a w + b.$$

Useful interpretations:

- `a = 1` and varying `b` gives translation in log space, which becomes multiplication by `exp(b)` in the `z`-plane.
- Real `a > 1` behaves like a power map.
- Nonzero imaginary part in `a` mixes radial and angular information.

In [ ]:
a = 1.0 + 1.0j
b = 0.45 + 0.80j

operated_log = cip.render_log_plane_preview(
    image,
    origin_px,
    pixels_per_unit,
    output_shape=(360, 480),
    inner_cutoff_px=inner_cutoff_px,
    a=a,
    b=b,
    inverse_affine=True,
)

print(cip.describe_mapping(image.shape, origin_px, pixels_per_unit, a=a, b=b, inner_cutoff_px=inner_cutoff_px))

fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))
axes[0].imshow(log_preview["image"], extent=(u0, u1, v0, v1), origin="lower")
axes[0].set_title("Before applying aw + b")
axes[0].set_xlabel("Re(w)")
axes[0].set_ylabel("Im(w)")
axes[0].grid(color="white", alpha=0.15)

axes[1].imshow(operated_log["image"], extent=(u0, u1, v0, v1), origin="lower")
axes[1].set_title(f"After aw + b with a={a}, b={b}")
axes[1].set_xlabel("Re(w')")
axes[1].set_ylabel("Im(w')")
axes[1].grid(color="white", alpha=0.15)

plt.tight_layout()

## 4. Map Back With The Exponential

For each output-plane point `z'`, we compute the principal logarithm `w' = log(z')`, invert the affine step to recover `w = (w' - b)/a`, and then recover the source position through `z = exp(w)`.

This is the step where the branch cut becomes visible.

In [ ]:
output = cip.render_output_plane(
    image,
    origin_px,
    pixels_per_unit,
    a=a,
    b=b,
    output_shape=image.shape[:2],
)

fig, axes = plt.subplots(2, 2, figsize=(13, 10))

cip.plot_image(axes[0, 0], image, "Source z-plane")
cip.annotate_source_plane(axes[0, 0], image.shape, origin_px, pixels_per_unit)

axes[0, 1].imshow(log_preview["image"], extent=(u0, u1, v0, v1), origin="lower")
axes[0, 1].set_title("Principal log preview")
axes[0, 1].set_xlabel("Re(w)")
axes[0, 1].set_ylabel("Im(w)")

axes[1, 0].imshow(operated_log["image"], extent=(u0, u1, v0, v1), origin="lower")
axes[1, 0].set_title("Operated log preview")
axes[1, 0].set_xlabel("Re(w')")
axes[1, 0].set_ylabel("Im(w')")

cip.plot_image(axes[1, 1], output["image"], "Back in the z-plane after exp")
cip.annotate_source_plane(axes[1, 1], output["image"].shape, origin_px, pixels_per_unit)

plt.tight_layout()

## Suggested Experiments

- Keep `a = 1` and vary `b` to study how translation in log space becomes multiplication in the image plane.
- Try `a = 2 + 0j` to see a power-like effect.
- Try `a = 0.5 + 0j` to see a root-like compression.
- Give `a` a nonzero imaginary part to mix radius and angle.
- Move `origin_px` away from the image center and watch how the branch cut and spiral structure change.
- Replace the demo image with a photograph, logo, or diagram and compare which structures survive the branch cut best.